In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
!pip -q install openpyxl

#Merge new feature score

In [ ]:
EXCEL_PATH  = "/content/drive/MyDrive/OIL/Dataset/root_add_newfeature.xlsx"
DB_PATH     = "/content/drive/MyDrive/OIL/Dataset/news_featuresv4.db"
OUT_PATH   = "/content/drive/MyDrive/OIL/Dataset/root_add_newfeature_with_news3v2.xlsx"

In [ ]:
import pandas as pd
import sqlite3

In [ ]:
def parse_excel_date(s: pd.Series) -> pd.Series:
    if pd.api.types.is_numeric_dtype(s):
        dtv = pd.to_datetime(s, unit="D", origin="1899-12-30", errors="coerce")
    else:
        dtv = pd.to_datetime(s, errors="coerce")
    return dtv.dt.normalize()

In [ ]:
def published_to_local_day(published_at: pd.Series, tz: str = "Asia/Ho_Chi_Minh") -> pd.Series:
    ts = pd.to_datetime(published_at, errors="coerce")
    try:
        if getattr(ts.dt, "tz", None) is not None:
            ts = ts.dt.tz_convert(tz)
        else:
            ts = ts.dt.tz_localize(tz)
    except Exception:
        ts = pd.to_datetime(published_at, errors="coerce", utc=True).dt.tz_convert(tz)
    return ts.dt.normalize().dt.tz_localize(None)

In [ ]:
def agg_one_day(g: pd.DataFrame, shock_abs=20.0, shock_abnormal=0.8):
    imp = pd.to_numeric(g["impact_score"], errors="coerce").dropna()
    abnormal_max = float(pd.to_numeric(g["news_abnormal"], errors="coerce").fillna(0).max())

    if imp.empty:
        impact_base = 0.0
        shock_val = 0.0
    else:
        q25, q50, q75 = imp.quantile([0.25, 0.50, 0.75]).tolist()
        impact_base = float((q25 + q50 + q75) / 3.0)
        shock_val = float(imp.loc[imp.abs().idxmax()])  # max-abs, giữ dấu

    use_shock = (abs(shock_val) >= float(shock_abs)) or (abnormal_max >= float(shock_abnormal))
    impact = shock_val if use_shock else impact_base

    if impact > 0:
        trend = "INCREASE"
    elif impact < 0:
        trend = "DECREASE"
    else:
        trend = "NEUTRAL"

    return pd.Series({
        "news_abnormal": abnormal_max,
        "impact_score": float(impact),
        "trend": trend,
    })

In [ ]:
def build_daily_3_features(db_path: str,
                           tz="Asia/Ho_Chi_Minh",
                           shift_days=0,
                           shock_abs=20.0,
                           shock_abnormal=0.8) -> pd.DataFrame:
    con = sqlite3.connect(db_path)
    news = pd.read_sql_query("""
        SELECT published_at, news_abnormal, impact_score, trend, gemini_status
        FROM raw_news
        WHERE gemini_status = 200
          AND published_at IS NOT NULL AND TRIM(published_at) <> '';
    """, con)
    con.close()

    if news.empty:
        return pd.DataFrame(columns=["Ngày","news_abnormal","impact_score","trend"])

    news["day"] = published_to_local_day(news["published_at"], tz=tz)
    news = news.dropna(subset=["day"]).copy()

    if shift_days != 0:
        news["day"] = news["day"] + pd.Timedelta(days=int(shift_days))

    daily = (
        news.groupby("day", sort=True)
            .apply(lambda g: agg_one_day(g, shock_abs=shock_abs, shock_abnormal=shock_abnormal))
            .reset_index()
            .rename(columns={"day":"Ngày"})
    )
    daily["Ngày"] = parse_excel_date(daily["Ngày"])
    return daily

In [ ]:
def merge_to_excel(excel_path: str,
                   db_path: str,
                   out_path: str,
                   date_col="Ngày",
                   tz="Asia/Ho_Chi_Minh",
                   shift_days=0,
                   shock_abs=20.0,
                   shock_abnormal=0.8):
    df = pd.read_excel(excel_path, engine="openpyxl")
    if date_col not in df.columns:
        raise ValueError(f"Không thấy cột '{date_col}'. Cột hiện có: {list(df.columns)}")

    df[date_col] = parse_excel_date(df[date_col])

    daily = build_daily_3_features(
        db_path=db_path,
        tz=tz,
        shift_days=shift_days,
        shock_abs=shock_abs,
        shock_abnormal=shock_abnormal
    )

    out = df.merge(daily, how="left", left_on=date_col, right_on="Ngày")
    if "Ngày_y" in out.columns:  # phòng khi merge tạo cột trùng
        out = out.drop(columns=["Ngày_y"]).rename(columns={"Ngày_x":"Ngày"})

    out["news_abnormal"] = out["news_abnormal"].fillna(0.0)
    out["impact_score"]  = out["impact_score"].fillna(0.0)
    out["trend"]         = out["trend"].fillna("NEUTRAL")

    out.to_excel(out_path, index=False, engine="openpyxl")
    return out_path

In [ ]:
saved = merge_to_excel(
    excel_path=EXCEL_PATH,
    db_path=DB_PATH,
    out_path=OUT_PATH,
    shift_days=1,
    shock_abs=20.0,
    shock_abnormal=0.8
)

/tmp/ipython-input-2020003969.py:26: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: agg_one_day(g, shock_abs=shock_abs, shock_abnormal=shock_abnormal))


# Train model with dataset_new_score

##Import

In [ ]:
import os, json, time, math, argparse
from dataclasses import dataclass
from typing import List, Tuple, Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset

In [ ]:
TARGET_COLS = ["MG95", "MG92", "DO 0.001%", "DO 0.05%"]

##Utils

In [ ]:
# @title
def set_seed(seed: int = 42):
    import random
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    # set benchmark=True giúp nhanh hơn trên GPU
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True

def norm_cols(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = (
        df.columns.astype(str)
          .str.replace(r"\s+", " ", regex=True)
          .str.strip()
    )
    return df

def add_calendar_features(df: pd.DataFrame, date_col: str) -> pd.DataFrame:
    df = df.copy()
    d = pd.to_datetime(df[date_col])
    df["dow"] = d.dt.weekday.astype(float)
    df["month"] = d.dt.month.astype(float)
    df["year"] = d.dt.year.astype(float)
    df["dom"] = d.dt.day.astype(float)
    return df

def prep_news(df: pd.DataFrame, shift_days: int = 1) -> pd.DataFrame:
    df = df.copy()
    if "news_abnormal" not in df.columns:
        df["news_abnormal"] = 0.0
    if "impact_score" not in df.columns:
        df["impact_score"] = 0.0

    if shift_days and shift_days > 0:
        df["news_abnormal"] = df["news_abnormal"].shift(shift_days).fillna(0.0)
        df["impact_score"]  = df["impact_score"].shift(shift_days).fillna(0.0)
    return df

def fill_missing(df: pd.DataFrame, date_col: str) -> pd.DataFrame:
    df = df.copy()
    for c in df.columns:
        if c == date_col:
            continue
        if pd.api.types.is_numeric_dtype(df[c]):
            df[c] = pd.to_numeric(df[c], errors="coerce")
            df[c] = df[c].ffill().bfill()
    return df

def time_split(T: int, K: int, val_size: int, test_size: int):
    te_e = T
    te_s = max(0, T - test_size)
    va_e = te_s
    va_s = max(0, va_e - val_size)
    tr_s = 0
    tr_e = va_s
    tr_e = max(tr_e, K + 1)
    return (tr_s, tr_e), (va_s, va_e), (te_s, te_e)

def mape(y_pred: np.ndarray, y_true: np.ndarray, eps: float = 1e-6) -> float:
    if y_pred is None or y_true is None or len(y_pred) == 0:
        return float("nan")
    y_pred = np.asarray(y_pred, dtype=np.float32)
    y_true = np.asarray(y_true, dtype=np.float32)
    return float(np.mean(np.abs(y_pred - y_true) / (np.abs(y_true) + eps)) * 100.0)

def fit_linear_calibrator(Yp: np.ndarray, Yt: np.ndarray, eps: float = 1e-8):
    Yp = np.asarray(Yp, dtype=np.float32)
    Yt = np.asarray(Yt, dtype=np.float32)
    a = np.zeros((Yp.shape[1],), dtype=np.float32)
    b = np.zeros((Yp.shape[1],), dtype=np.float32)
    for j in range(Yp.shape[1]):
        x = Yp[:, j]
        y = Yt[:, j]
        vx = float(np.var(x))
        if vx < eps:
            a[j] = 1.0
            b[j] = float(np.mean(y) - np.mean(x))
        else:
            cov = float(np.mean((x - np.mean(x)) * (y - np.mean(y))))
            a[j] = cov / (vx + eps)
            b[j] = float(np.mean(y) - a[j] * np.mean(x))
    return a, b

def best_blend_alpha_mae(Y_cal: np.ndarray, Y_naive: np.ndarray, Y_true: np.ndarray, step: float = 0.01):
    Y_cal = np.asarray(Y_cal, dtype=np.float32)
    Y_naive = np.asarray(Y_naive, dtype=np.float32)
    Y_true = np.asarray(Y_true, dtype=np.float32)

    best_alpha = 0.0
    best_mae = float("inf")
    a = 0.0
    while a <= 1.000001:
        Yb = a * Y_cal + (1.0 - a) * Y_naive
        mae = float(np.mean(np.abs(Yb - Y_true)))
        if mae < best_mae:
            best_mae = mae
            best_alpha = a
        a += step
    return float(best_alpha), float(best_mae)

def next_business_days(last_date: pd.Timestamp, n: int = 5) -> List[pd.Timestamp]:
    days = []
    d = pd.Timestamp(last_date)
    while len(days) < n:
        d = d + pd.Timedelta(days=1)
        if d.weekday() < 5:
            days.append(d)
    return days

## Dataset

In [ ]:
# @title
class WindowDeltaDataset(Dataset):
    def __init__(self, Xs: np.ndarray, Ys: np.ndarray, K: int, start: int, end: int, step: int = 1):
        self.Xs = Xs
        self.Ys = Ys
        self.K = int(K)
        self.step = max(1, int(step))
        self.t_list = []
        t0 = max(self.K, start)
        for t in range(t0, end, self.step):
            self.t_list.append(t)

    def __len__(self):
        return len(self.t_list)

    def __getitem__(self, i):
        t = self.t_list[i]
        xb = self.Xs[t-self.K:t]   # [K,D]
        yb = self.Ys[t]            # [4]
        return torch.from_numpy(xb).float(), torch.from_numpy(yb).float()


## Model

In [ ]:
# @title
class HybridTriNet(nn.Module):
    def __init__(
        self,
        k: int, D_in: int, H: int, D_out: int,
        d_feat: int = 64, kan_depth: int = 1, kan_drop: float = 0.35,
        gru_hidden: int = 96, gru_layers: int = 1, gru_drop: float = 0.25,
        attn_dmodel: int = 48, attn_heads: int = 4, attn_layers: int = 1, attn_drop: float = 0.35,
        patch_len: int = 16, stride: int = 8,
    ):
        super().__init__()
        self.in_norm = nn.LayerNorm(D_in)

        # (A) GRU branch
        self.gru = nn.GRU(
            input_size=D_in,
            hidden_size=gru_hidden,
            num_layers=gru_layers,
            batch_first=True,
            dropout=(gru_drop if gru_layers > 1 else 0.0),
            bidirectional=False,
        )

        # (B) Attention branch (TransformerEncoder)
        self.proj = nn.Linear(D_in, attn_dmodel)
        enc_layer = nn.TransformerEncoderLayer(
            d_model=attn_dmodel,
            nhead=attn_heads,
            dim_feedforward=max(128, attn_dmodel * 4),
            dropout=attn_drop,
            batch_first=True,
            activation="gelu",
        )
        self.attn = nn.TransformerEncoder(enc_layer, num_layers=attn_layers)

        # (C) Head
        self.head = nn.Sequential(
            nn.Linear(gru_hidden + attn_dmodel, 128),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(128, D_out),
        )

    def forward(self, x):
        x = self.in_norm(x)

        gru_out, _ = self.gru(x)     # [B,K,H]
        h_gru = gru_out[:, -1, :]    # [B,H]

        z = self.proj(x)            # [B,K,d]
        z = self.attn(z)            # [B,K,d]
        h_attn = z[:, -1, :]        # [B,d]

        h = torch.cat([h_gru, h_attn], dim=-1)
        return self.head(h)

## EMA

In [ ]:
# @title
class EMA:
    def __init__(self, model, decay=0.999):
        self.decay = float(decay)
        self.shadow = {}
        self.backup = None
        self.reset(model)

    @torch.no_grad()
    def reset(self, model):
        self.shadow = {}
        for k, v in model.state_dict().items():
            if torch.is_tensor(v):
                self.shadow[k] = v.detach().clone()

    @torch.no_grad()
    def update(self, model):
        for k, v in model.state_dict().items():
            if k not in self.shadow:
                self.shadow[k] = v.detach().clone()
            else:
                self.shadow[k].mul_(self.decay).add_(v.detach(), alpha=(1.0 - self.decay))

    @torch.no_grad()
    def apply(self, model):
        self.backup = {}
        sd = model.state_dict()
        for k, v in sd.items():
            if torch.is_tensor(v):
                self.backup[k] = v.detach().clone()
                v.copy_(self.shadow[k])

    @torch.no_grad()
    def restore(self, model):
        if self.backup is None:
            return
        sd = model.state_dict()
        for k, v in sd.items():
            if torch.is_tensor(v) and k in self.backup:
                v.copy_(self.backup[k])
        self.backup = None

## Collect preds

In [ ]:
# @title
@torch.no_grad()
def collect_price_preds(
    model,
    Xs: np.ndarray,
    df: pd.DataFrame,
    feature_cols: List[str],
    K: int,
    start: int,
    end: int,
    device: str,
    x_mean_np: np.ndarray,
    x_std_np: np.ndarray,
    y_mean_np: np.ndarray,
    y_std_np: np.ndarray,
) -> Tuple[np.ndarray, np.ndarray]:
    tgt_idx_list = [feature_cols.index(c) for c in TARGET_COLS]

    x_mean_t = torch.tensor(x_mean_np[tgt_idx_list], device=device)
    x_std_t  = torch.tensor(x_std_np[tgt_idx_list], device=device)
    y_mean_t = torch.tensor(y_mean_np, device=device)
    y_std_t  = torch.tensor(y_std_np, device=device)

    preds, trues = [], []
    t0 = max(K, start)
    for t in range(t0, end):
        xb = torch.from_numpy(Xs[t-K:t]).float().unsqueeze(0).to(device)
        pred_s = model(xb)  # scaled delta

        xb_targets = xb[:, -1, tgt_idx_list]          # price(t-1) normalized
        last_price = xb_targets * x_std_t + x_mean_t  # unnormalize

        pred_d = pred_s * y_std_t + y_mean_t
        pred_p = last_price + pred_d

        true_p = torch.tensor(df.loc[t, TARGET_COLS].values.astype(np.float32), device=device).unsqueeze(0)

        preds.append(pred_p.squeeze(0).detach().cpu().numpy())
        trues.append(true_p.squeeze(0).detach().cpu().numpy())

    if len(preds) == 0:
        return np.zeros((0,4), np.float32), np.zeros((0,4), np.float32)
    return np.stack(preds, axis=0).astype(np.float32), np.stack(trues, axis=0).astype(np.float32)


## Forecast

In [ ]:
# @title
@torch.no_grad()
def forecast_5_business_days(
    model,
    df_used: pd.DataFrame,
    feature_cols: List[str],
    K: int,
    device: str,
    x_mean_np: np.ndarray,
    x_std_np: np.ndarray,
    y_mean_np: np.ndarray,
    y_std_np: np.ndarray,
    date_col: str = "Ngày",
    shift_news_days: int = 1,
    cal_a: Optional[np.ndarray] = None,
    cal_b: Optional[np.ndarray] = None,
    blend_alpha: Optional[float] = None,
):
    model.eval()
    last_date = pd.to_datetime(df_used[date_col].iloc[-1])
    fut_dates = next_business_days(last_date, 5)

    tgt_idx_list = [feature_cols.index(c) for c in TARGET_COLS]
    x_mean_t = torch.tensor(x_mean_np[tgt_idx_list], device=device)
    x_std_t  = torch.tensor(x_std_np[tgt_idx_list], device=device)
    y_mean_t = torch.tensor(y_mean_np, device=device)
    y_std_t  = torch.tensor(y_std_np, device=device)

    hist = df_used.copy()
    outs = []

    for d in fut_dates:
        row = hist.iloc[-1].copy()
        row[date_col] = d

        # calendar
        row["dow"] = float(d.weekday())
        row["month"] = float(d.month)
        row["year"] = float(d.year)
        row["dom"] = float(d.day)

        # future news unknown
        if "news_abnormal" in row.index: row["news_abnormal"] = 0.0
        if "impact_score" in row.index:  row["impact_score"]  = 0.0

        # build last K window
        X_hist = hist[feature_cols].values.astype(np.float32)
        Xs_hist = (X_hist - x_mean_np) / x_std_np
        xb = torch.from_numpy(Xs_hist[-K:]).float().unsqueeze(0).to(device)

        pred_s = model(xb)
        xb_targets = xb[:, -1, tgt_idx_list]
        last_price = xb_targets * x_std_t + x_mean_t

        pred_d = pred_s * y_std_t + y_mean_t
        pred_raw = (last_price + pred_d).squeeze(0).detach().cpu().numpy()

        # calibrate
        if cal_a is not None and cal_b is not None:
            pred_cal = pred_raw * cal_a + cal_b
        else:
            pred_cal = pred_raw

        # naive = last known price
        naive = hist.iloc[-1][TARGET_COLS].values.astype(np.float32)

        # blend
        if blend_alpha is None:
            pred_final = pred_cal
        else:
            pred_final = float(blend_alpha) * pred_cal + (1.0 - float(blend_alpha)) * naive

        out = {"Ngày": d}
        for i, c in enumerate(TARGET_COLS):
            out[c + "_pred"] = float(pred_final[i])
            row[c] = float(pred_final[i])  # autoregressive update

        outs.append(out)
        hist = pd.concat([hist, pd.DataFrame([row])], ignore_index=True)

    return pd.DataFrame(outs)


## Config

In [ ]:
# @title
@dataclass
class CFG:
    excel_path: str
    out_dir: str
    date_col: str = "Ngày"

    K: int = 32
    STEP: int = 1
    val_size: int = 60
    test_size: int = 22

    batch_size: int = 64
    epochs: int = 120

    lr: float = 5e-4
    weight_decay: float = 1e-4

    lr_factor: float = 0.5
    lr_patience: int = 12
    lr_threshold: float = 1e-4
    min_lr: float = 1e-6

    patience: int = 20
    min_delta: float = 1e-4

    shift_news_days: int = 1

    warmup_epochs: int = 3
    noise_std: float = 0.0
    feature_dropout: float = 0.0

    loss_alpha: float = 0.2
    loss_mape_weight: float = 1.0
    mape_eps: float = 1e-3
    clip_grad: float = 5.0

##MAIN TRAIN LOOP

In [ ]:
# @title
def train_and_forecast(cfg: CFG, mape_every: int = 5):
    set_seed(42)
    os.makedirs(cfg.out_dir, exist_ok=True)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print("Device:", device)

    ckpt_path = os.path.join(cfg.out_dir, "best_state_dict.pth")
    meta_path = os.path.join(cfg.out_dir, "meta.json")
    log_path  = os.path.join(cfg.out_dir, "train_log.csv")

    df = pd.read_excel(cfg.excel_path, engine="openpyxl")
    df = norm_cols(df)

    if cfg.date_col not in df.columns:
        raise ValueError(f"Không có cột '{cfg.date_col}'")

    df[cfg.date_col] = pd.to_datetime(df[cfg.date_col], errors="coerce")
    df = df.dropna(subset=[cfg.date_col]).sort_values(cfg.date_col).reset_index(drop=True)

    miss = [c for c in TARGET_COLS if c not in df.columns]
    if miss:
        raise ValueError(f"Thiếu target: {miss}")

    df = add_calendar_features(df, cfg.date_col)
    df = prep_news(df, shift_days=cfg.shift_news_days)
    df = fill_missing(df, cfg.date_col)

    for c in TARGET_COLS:
        df[c] = df[c].ffill().bfill()

    Yd = np.stack([df[c].diff().fillna(0.0).values.astype(np.float32) for c in TARGET_COLS], axis=1)

    feature_cols = []
    for c in df.columns:
        if c in [cfg.date_col, "trend"]:
            continue
        if pd.api.types.is_numeric_dtype(df[c]):
            feature_cols.append(c)
    for c in TARGET_COLS:
        if c not in feature_cols:
            feature_cols.append(c)

    X = df[feature_cols].values.astype(np.float32)
    T, D_in = X.shape
    tr, va, te = time_split(T, cfg.K, cfg.val_size, cfg.test_size)
    tr_s, tr_e = tr; va_s, va_e = va; te_s, te_e = te
    print(f"T={T} | D_in={D_in} | K={cfg.K} | train={tr_e-tr_s} val={va_e-va_s} test={te_e-te_s}")

    if tr_e <= tr_s or va_e <= va_s or te_e <= te_s:
        raise RuntimeError("Không đủ dữ liệu, giảm K/val/test hoặc thêm dữ liệu")

    fit_end = tr_e
    x_mean_np = X[:fit_end].mean(axis=0).astype(np.float32)
    x_std_np  = (X[:fit_end].std(axis=0) + 1e-8).astype(np.float32)
    Xs = (X - x_mean_np) / x_std_np

    y_mean_np = Yd[:fit_end].mean(axis=0).astype(np.float32)
    y_std_np  = (Yd[:fit_end].std(axis=0) + 1e-8).astype(np.float32)
    Ys = (Yd - y_mean_np) / y_std_np
    ds_tr = WindowDeltaDataset(Xs, Ys, cfg.K, tr_s, tr_e, cfg.STEP)
    ds_va = WindowDeltaDataset(Xs, Ys, cfg.K, va_s, va_e, cfg.STEP)

    dl_tr = DataLoader(ds_tr, batch_size=cfg.batch_size, shuffle=True,  num_workers=0)
    dl_va = DataLoader(ds_va, batch_size=cfg.batch_size, shuffle=False, num_workers=0)

    tgt_idx_list = [feature_cols.index(c) for c in TARGET_COLS]

    # tensors để unnormalize
    x_mean_targets = torch.tensor(x_mean_np[tgt_idx_list], device=device)
    x_std_targets  = torch.tensor(x_std_np[tgt_idx_list],  device=device)
    y_mean_targets = torch.tensor(y_mean_np, device=device)
    y_std_targets  = torch.tensor(y_std_np,  device=device)

    # ---------- Model / Optim / Scheduler ----------
    model = HybridTriNet(
        k=cfg.K, D_in=D_in, H=1, D_out=4,
        gru_hidden=96, gru_layers=1,
        attn_dmodel=48, attn_heads=4, attn_layers=1,
    ).to(device)

    ema = EMA(model, decay=0.999)
    opt = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        opt, mode="min", factor=cfg.lr_factor, patience=cfg.lr_patience,
        threshold=cfg.lr_threshold, min_lr=cfg.min_lr,
    )

    # ---------- Tracking best ----------
    best_val_mape = float("inf")
    best_epoch = 0
    bad = 0
    logs = []

    va_mape_ema = None
    ema_beta = 0.90

    tr_mape_ema_last = float("nan")
    va_mape_last = float("nan")

    for epoch in range(cfg.epochs):
        model.train()

        # Warmup LR
        if epoch < cfg.warmup_epochs and cfg.warmup_epochs > 0:
            lr_w = cfg.lr * (epoch + 1) / max(1, cfg.warmup_epochs)
            for pg in opt.param_groups:
                pg["lr"] = lr_w

        t0 = time.time()
        mix_sum = 0.0
        delta_sum = 0.0
        mapeL_sum = 0.0
        n = 0

        for xb, yb in dl_tr:
            xb = xb.to(device)
            yb = yb.to(device)

            # augmentation (optional)
            if cfg.feature_dropout > 0:
                mask = (torch.rand_like(xb) > cfg.feature_dropout).float()
                xb = xb * mask
            if cfg.noise_std > 0:
                xb = xb + cfg.noise_std * torch.randn_like(xb)

            pred_s = model(xb)  # scaled delta

            # loss 1: delta MAE (scaled)
            delta_loss = torch.mean(torch.abs(pred_s - yb))

            # loss 2: price-level MAPE (1-step)
            xb_targets = xb[:, -1, tgt_idx_list]
            last_price = xb_targets * x_std_targets + x_mean_targets

            pred_d = pred_s * y_std_targets + y_mean_targets
            true_d = yb     * y_std_targets + y_mean_targets

            pred_p = last_price + pred_d
            true_p = last_price + true_d

            mape_loss = torch.mean(torch.abs(pred_p - true_p) / (torch.abs(true_p) + cfg.mape_eps))

            # mixed
            mix_loss = cfg.loss_alpha * delta_loss + cfg.loss_mape_weight * mape_loss

            opt.zero_grad(set_to_none=True)
            mix_loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.clip_grad)
            opt.step()
            ema.update(model)

            bs = xb.size(0)
            mix_sum += mix_loss.item() * bs
            delta_sum += delta_loss.item() * bs
            mapeL_sum += mape_loss.item() * bs
            n += bs

        train_mix = mix_sum / max(1, n)
        train_delta_scaled = delta_sum / max(1, n)
        train_mapeL = mapeL_sum / max(1, n)

        ema.apply(model)
        model.eval()

        v_mix_sum = 0.0
        v_delta_sum = 0.0
        v_mapeL_sum = 0.0
        v_mae_delta_sum = 0.0
        vn = 0

        with torch.no_grad():
            for xb, yb in dl_va:
                xb = xb.to(device)
                yb = yb.to(device)

                pred_s = model(xb)
                delta_loss = torch.mean(torch.abs(pred_s - yb))

                xb_targets = xb[:, -1, tgt_idx_list]
                last_price = xb_targets * x_std_targets + x_mean_targets

                pred_d = pred_s * y_std_targets + y_mean_targets
                true_d = yb     * y_std_targets + y_mean_targets

                pred_p = last_price + pred_d
                true_p = last_price + true_d

                mape_loss = torch.mean(torch.abs(pred_p - true_p) / (torch.abs(true_p) + cfg.mape_eps))
                mix_loss  = cfg.loss_alpha * delta_loss + cfg.loss_mape_weight * mape_loss

                mae_delta = torch.mean(torch.abs(pred_d - true_d))

                bs = xb.size(0)
                v_mix_sum += mix_loss.item() * bs
                v_delta_sum += delta_loss.item() * bs
                v_mapeL_sum += mape_loss.item() * bs
                v_mae_delta_sum += mae_delta.item() * bs
                vn += bs

        val_mix = v_mix_sum / max(1, vn)
        val_delta_scaled = v_delta_sum / max(1, vn)
        val_mapeL = v_mapeL_sum / max(1, vn)
        val_mae_delta = v_mae_delta_sum / max(1, vn)
        Yp_va, Yt_va = collect_price_preds(model, Xs, df, feature_cols, cfg.K, va_s, va_e, device,
                                           x_mean_np, x_std_np, y_mean_np, y_std_np)
        va_mape_last = mape(Yp_va, Yt_va)

        if (epoch == 0) or ((epoch + 1) % mape_every == 0):
            Yp_tr, Yt_tr = collect_price_preds(model, Xs, df, feature_cols, cfg.K, tr_s, tr_e, device,
                                               x_mean_np, x_std_np, y_mean_np, y_std_np)
            tr_mape_ema_last = mape(Yp_tr, Yt_tr)

        # smooth metric cho scheduler
        if np.isfinite(va_mape_last):
            va_mape_ema = float(va_mape_last) if va_mape_ema is None else (ema_beta * va_mape_ema + (1.0 - ema_beta) * float(va_mape_last))
        else:
            va_mape_ema = float("inf")

        metric = va_mape_last if np.isfinite(va_mape_last) else float("inf")

        if metric < best_val_mape - cfg.min_delta:
            best_val_mape = metric
            best_epoch = epoch + 1
            bad = 0

            torch.save(model.state_dict(), ckpt_path)
            meta = {
                "cfg": cfg.__dict__,
                "feature_cols": feature_cols,
                "x_mean": x_mean_np.tolist(),
                "x_std": x_std_np.tolist(),
                "y_mean": y_mean_np.tolist(),
                "y_std": y_std_np.tolist(),
                "date_col": cfg.date_col,
                "target_cols": TARGET_COLS,
                "D_in": int(D_in),
                "splits": {"train": [tr_s, tr_e], "val": [va_s, va_e], "test": [te_s, te_e]},
                "best_val_mape": float(best_val_mape),
                "best_epoch": int(best_epoch),
            }
            with open(meta_path, "w", encoding="utf-8") as f:
                json.dump(meta, f, ensure_ascii=False, indent=2)
        else:
            bad += 1

        # restore weights để tiếp tục train
        ema.restore(model)

        # scheduler sau warmup
        lr_before = float(opt.param_groups[0]["lr"])
        if epoch >= cfg.warmup_epochs and np.isfinite(va_mape_ema):
            scheduler.step(va_mape_ema)
        lr_after = float(opt.param_groups[0]["lr"])
        lr_dropped = (lr_after < lr_before - 1e-12)
        lr_msg = "  [LR↓]" if lr_dropped else ""
        lr_now = float(opt.param_groups[0]["lr"])

        # log
        logs.append({
            "epoch": epoch + 1,
            "lr": lr_now,
            "train_mix": float(train_mix),
            "train_delta_scaled": float(train_delta_scaled),
            "train_mapeL": float(train_mapeL),
            "val_mix": float(val_mix),
            "val_delta_scaled": float(val_delta_scaled),
            "val_mapeL": float(val_mapeL),
            "val_mae_delta": float(val_mae_delta),
            "train_mape_ema": float(tr_mape_ema_last) if np.isfinite(tr_mape_ema_last) else None,
            "val_mape": float(va_mape_last) if np.isfinite(va_mape_last) else None,
            "val_mape_ema": float(va_mape_ema) if np.isfinite(va_mape_ema) else None,
            "lr_dropped": bool(lr_dropped),
            "best_val_mape_sofar": float(best_val_mape) if np.isfinite(best_val_mape) else None,
            "epoch_time_s": float(time.time() - t0),
        })
        pd.DataFrame(logs).to_csv(log_path, index=False)

        print(
            f"Epoch {epoch+1:03d} | lr={lr_now:.2e}{lr_msg} | "
            f"train_loss={train_mix:.5f}, mapeL={train_mapeL:.5f}) | "
            f"val_loss={val_mix:.5f}, mapeL={val_mapeL:.5f}) | "
            f"valΔMAE={val_mae_delta:.4f} | "
            f"trainMAPE={tr_mape_ema_last:.3f} | valMAPE={va_mape_last:.3f} | valMAPE_ema={va_mape_ema:.3f}"
        )

        if bad >= cfg.patience:
            print(f"Early stopping. Best valMAPE={best_val_mape:.3f}% at epoch {best_epoch}.")
            break
    model.load_state_dict(torch.load(ckpt_path, map_location=device))

    Yp_val, Yt_val = collect_price_preds(model, Xs, df, feature_cols, cfg.K, va_s, va_e, device,
                                         x_mean_np, x_std_np, y_mean_np, y_std_np)
    Yp_te, Yt_te = collect_price_preds(model, Xs, df, feature_cols, cfg.K, te_s, te_e, device,
                                       x_mean_np, x_std_np, y_mean_np, y_std_np)

    cal_a, cal_b = fit_linear_calibrator(Yp_val, Yt_val)

    start_t = max(cfg.K, va_s)
    naive_idx = list(range(start_t - 1, va_e - 1))
    true_idx  = list(range(start_t, va_e))
    Y_naive = df.loc[naive_idx, TARGET_COLS].values.astype(np.float32)
    Y_true  = df.loc[true_idx,  TARGET_COLS].values.astype(np.float32)

    m = min(len(Y_naive), len(Y_true), len(Yp_val))
    Y_naive = Y_naive[-m:]
    Y_true  = Y_true[-m:]
    Y_cal   = (Yp_val[-m:] * cal_a + cal_b)

    alpha, val_price_mae = best_blend_alpha_mae(Y_cal, Y_naive, Y_true)

    val_price_mape_raw = mape(Yp_val, Yt_val)
    val_price_mape_cal = mape(Yp_val * cal_a + cal_b, Yt_val)
    val_price_mape_bl  = mape(alpha*(Yp_val * cal_a + cal_b) + (1-alpha)*Y_naive, Y_true)

    te_price_mape_raw = mape(Yp_te, Yt_te)
    te_price_mape_cal = mape(Yp_te * cal_a + cal_b, Yt_te)

    # print("Calibration a:", cal_a, "b:", cal_b)
    # print("Blend alpha:", alpha)
    # print("VAL MAE (blend):", float(val_price_mae))
    # print("VAL MAPE raw/cal/blend:", float(val_price_mape_raw), float(val_price_mape_cal), float(val_price_mape_bl))
    # print("TEST MAPE raw/cal:", float(te_price_mape_raw), float(te_price_mape_cal))

    pred5 = forecast_5_business_days(
        model=model,
        df_used=df,
        feature_cols=feature_cols,
        K=cfg.K,
        device=device,
        x_mean_np=x_mean_np,
        x_std_np=x_std_np,
        y_mean_np=y_mean_np,
        y_std_np=y_std_np,
        date_col=cfg.date_col,
        shift_news_days=cfg.shift_news_days,
        cal_a=cal_a,
        cal_b=cal_b,
        blend_alpha=alpha
    )

    pred_path = os.path.join(cfg.out_dir, "forecast_5_business_days.xlsx")
    pred5.to_excel(pred_path, index=False)

    # # plots
    # plot_training_curves(log_path, cfg.out_dir)

    # print("Saved plots to:", cfg.out_dir)
    # print("Saved best:", ckpt_path)
    # print("Saved meta:", meta_path)
    # print("Saved log:", log_path)
    # print("Saved forecast:", pred_path)
    return pred5

if __name__ == "__main__":
    EXCEL_PATH = "/content/drive/MyDrive/OIL/Dataset/root_add_newfeature_with_news3v2.xlsx"
    OUT_DIR    = "/content/drive/MyDrive/OIL/Output/hybridtrinet_news_delta_best_v2"

    cfg = CFG(
        excel_path=EXCEL_PATH,
        out_dir=OUT_DIR,
        date_col="Ngày",
        K=32,
        val_size=60,
        test_size=22,
        epochs=120,
        batch_size=64,
        lr=5e-4,
    )
    train_and_forecast(cfg)

Device: cuda
T=475 | D_in=22 | K=32 | train=393 val=60 test=22
Epoch 001 | lr=1.67e-04 | train_loss=0.15911, mapeL=0.01272) | val_loss=0.13161, mapeL=0.01096) | valΔMAE=0.9235 | trainMAPE=1.275 | valMAPE=1.096 | valMAPE_ema=1.096
Epoch 002 | lr=3.33e-04 | train_loss=0.15833, mapeL=0.01266) | val_loss=0.13160, mapeL=0.01096) | valΔMAE=0.9233 | trainMAPE=1.275 | valMAPE=1.096 | valMAPE_ema=1.096
Epoch 003 | lr=5.00e-04 | train_loss=0.15799, mapeL=0.01263) | val_loss=0.13157, mapeL=0.01096) | valΔMAE=0.9232 | trainMAPE=1.275 | valMAPE=1.096 | valMAPE_ema=1.096
Epoch 004 | lr=5.00e-04 | train_loss=0.15709, mapeL=0.01256) | val_loss=0.13154, mapeL=0.01096) | valΔMAE=0.9230 | trainMAPE=1.275 | valMAPE=1.096 | valMAPE_ema=1.096
Epoch 005 | lr=5.00e-04 | train_loss=0.15627, mapeL=0.01249) | val_loss=0.13151, mapeL=0.01095) | valΔMAE=0.9228 | trainMAPE=1.274 | valMAPE=1.095 | valMAPE_ema=1.096
Epoch 006 | lr=5.00e-04 | train_loss=0.15506, mapeL=0.01240) | val_loss=0.13149, mapeL=0.01095) | valΔ